# B2-023 — Session 2: Training and Diagnosing GANs

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

Session 1 described the game; this session plays it with gradient descent.
It fixes the alternating-update protocol the unit uses everywhere (pinned helpers `d_step` and `g_step`, separate optimizers, `detach` discipline), explains why loss traces are not a measure of sample quality, and builds the mode-coverage diagnostic that the known four-blob mixture makes possible.
It closes with fixed-seed training traces that show the two classic failures: mode collapse and an overpowering discriminator.
Practices after this session: p07, p08, p14, p17, p22.

## 1. Alternating updates with two optimizers

A GAN has two parameter sets and two objectives, so it has **two optimizers**: `opt_d` holds only the discriminator's parameters and `opt_g` holds only the generator's. One training **step** is a discriminator update followed by a generator update. This unit pins both halves as helper functions so that every practice (p07, p17) and every test can drive them separately:

```python
def d_step(G, D, opt_d, real, z):
    opt_d.zero_grad(set_to_none=True)
    loss = discriminator_loss(D(real), D(G(z).detach()))
    loss.backward()
    opt_d.step()
    return loss.item()

def g_step(G, D, opt_g, z):
    opt_g.zero_grad(set_to_none=True)
    loss = generator_loss(D(G(z)))
    loss.backward()
    opt_g.step()
    return loss.item()
```

`gan_step(G, D, opt_g, opt_d, real, z)` calls `d_step(G, D, opt_d, real, z)` and then `g_step(G, D, opt_g, z)`, and returns both losses. Each helper begins with `zero_grad(set_to_none=True)` (Book 1 `torch-optimizers`), so stale gradients from the other player's step are discarded rather than added in.

**Noise is an argument.** The noise batch `z` is drawn once per step by the training loop from its own seeded `torch.Generator` (Book 1 `random-seeding`) and passed to both helpers. Nothing inside a helper draws random numbers, so each helper is a deterministic function of its inputs and can be tested with literal tensors.

**Full batch.** In this unit the real batch is all 256 training rows every step, and `z` has the same number of rows.

**Checkpoint 1A.** Which parameters does `opt_d.step()` change, and which does `opt_g.step()` change?

**Checkpoint 1B.** Why does the unit pass `z` into `d_step` and `g_step` instead of drawing it inside them?

**Collected answers.** `opt_d.step()` changes only the discriminator's parameters and `opt_g.step()` only the generator's, because each optimizer was constructed from one network's `parameters()`. So each helper is deterministic and testable with fixed tensors, and the whole run is reproducible from one dedicated noise generator instead of the global random state.

## 2. `detach` discipline

**In the discriminator step, cut the generator out of the graph.** The fake logits are computed from `G(z)`, which depends on the generator's parameters. Without `.detach()`, `loss.backward()` in the D-step computes gradients for **every** parameter in the graph, the generator's included. Those gradients are wasted work, and they sit in the generator's `.grad` fields. In this unit's protocol, `g_step` starts with `opt_g.zero_grad(set_to_none=True)`, which would discard them; but a pipeline that forgot that call would update the generator with the **discriminator's** gradient, pushing it in exactly the direction that helps the discriminator.

`fake = G(z).detach()` returns a tensor with the same values and no history (Book 1 `requires-grad`), so the D-step's backward pass stops at the discriminator.

**The probe.** Right after a correct `d_step`, every generator parameter's `.grad` is `None`: the generator was never part of the D-step's graph, and its gradients were last cleared with `set_to_none=True`. A missing `detach` makes those fields hold tensors. This is the check p07 and the unit's tests use; comparing generator **weights** before and after the D-step cannot detect the bug, because `opt_d` never touches generator weights in either case.

**In the generator step, do not detach.** The generator's gradient must flow **through** the discriminator: $\partial\mathcal L_G/\partial\theta=\frac{\partial\mathcal L_G}{\partial\text{logit}}\cdot\frac{\partial\text{logit}}{\partial G(z)}\cdot\frac{\partial G(z)}{\partial\theta}$. The discriminator's parameters receive gradients too, but `opt_g` does not hold them, so the discriminator does not change during the G-step; its stale gradients are cleared by the next `d_step`.

**Checkpoint 2A.** After a correct `d_step`, what is `G[0].weight.grad`?

**Checkpoint 2B.** During `g_step`, the discriminator's parameters receive `.grad` values. Are the discriminator's weights changed by that step? Why?

**Collected answers.** `None`. No: `opt_g` was built from the generator's parameters only, so `opt_g.step()` never reads or writes the discriminator's weights.

In [ ]:
import importlib.util
from pathlib import Path

import torch

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)
from torch import nn
import torch.nn.functional as F


def make_generator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))


def make_discriminator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))


def loss_d(real_logits, fake_logits):
    return (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))


def loss_g(fake_logits):
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


def run_gan(lr_g, lr_d, steps, noise_seed):
    """Lesson-scale alternating training on the 256 training rows; returns (G, D, trace)."""
    real = generative_data.train_tensor("mixture2d")
    torch.manual_seed(SEED)
    G, D = make_generator(), make_discriminator()
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d)
    noise = torch.Generator().manual_seed(noise_seed)
    trace = []
    for _ in range(steps):
        z = torch.randn(real.shape[0], 2, generator=noise)
        opt_d.zero_grad(set_to_none=True)                      # D-step
        d_value = loss_d(D(real), D(G(z).detach()))
        d_value.backward()
        opt_d.step()
        opt_g.zero_grad(set_to_none=True)                      # G-step
        g_value = loss_g(D(G(z)))
        g_value.backward()
        opt_g.step()
        trace.append((d_value.item(), g_value.item()))
    return G, D, trace


def coverage_counts(samples, centers, radius):
    d2 = ((samples[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)   # (n, K)
    nearest = d2.argmin(dim=1)                                           # ties -> lowest index
    inside = d2.min(dim=1).values <= radius ** 2
    return torch.bincount(nearest[inside], minlength=centers.shape[0])


def window_means(trace, width=50):
    return [tuple(round(sum(row[i] for row in trace[a:a + width]) / width, 3) for i in range(2))
            for a in range(0, len(trace), width * 2)]

torch.manual_seed(SEED)
G, D = make_generator(), make_discriminator()
opt_d = torch.optim.Adam(D.parameters(), lr=2e-3)
real = generative_data.train_tensor("mixture2d")[:8]
z = torch.randn(8, 2, generator=torch.Generator().manual_seed(SEED))

opt_d.zero_grad(set_to_none=True)
loss_d(D(real), D(G(z).detach())).backward()                   # correct D-step graph
print("with detach   : G[0].weight.grad is None ->", G[0].weight.grad is None)

opt_d.zero_grad(set_to_none=True)
loss_d(D(real), D(G(z))).backward()                            # missing detach
print("without detach: G[0].weight.grad is None ->", G[0].weight.grad is None)

## 3. Loss traces are not sample quality

In Book 1 (`autograd-training`) a falling training loss meant the model was fitting. A GAN breaks that habit: each player's loss is measured **against the other player**, and both players keep moving.

* The discriminator loss $\mathcal L_D$ hovers near $2\log2=1.386$ when $D$ cannot separate real from fake. That happens when the generator is good, but also when the discriminator is weak or has just been fooled by a temporary trick.
* The non-saturating generator loss $\mathcal L_G=-\text{mean}\log D(G(z))$ is low when the **current** discriminator is fooled. A generator that puts every sample on one blob the discriminator happens to accept can have a low $\mathcal L_G$.
* Neither loss is expected to decrease monotonically. Reading single steps is noise; this unit reads **window means** (for example, the mean over 50 consecutive steps).

So loss traces answer "is the game balanced?" and sample checks answer "is the generator good?". On the unit's known mixture the second question has a direct answer: count where the samples land (Section 4).

**Checkpoint 3A.** Why is a decreasing generator loss not a certificate of a good generator?

**Checkpoint 3B.** A run's 50-step window means of $\mathcal L_D$ are $1.27, 1.03, 1.31, 1.10$. Can you conclude that the generator reproduces all four blobs?

**Collected answers.** Because $\mathcal L_G$ only says the current discriminator is fooled; the discriminator may be weak, or the generator may fool it with a small part of the data distribution. No. These values say the game is roughly balanced, not which blobs are produced; only a sample check (mode coverage) answers that, and Section 6 shows a run with exactly such a trace that covers one blob.

## 4. Mode collapse and the mode-coverage diagnostic

**Mode collapse** is the failure in which the generator maps most noise vectors to one region (one "mode") or a few, ignoring the rest of the data distribution. It is rewarded by the game whenever the current discriminator accepts that region, and it is invisible in a single loss value.

**The diagnostic.** For a distribution with $K$ known component centres $c_1,\dots,c_K$ (here the four `mixture2d` centres), draw $n$ samples with fixed noise and apply this rule:

1. **Nearest centre.** Assign each sample $s$ to $k(s)=\arg\min_k\lVert s-c_k\rVert^2$; on an exact tie the **lowest index** wins (this is what `torch.argmin` returns).
2. **Radius.** Count the sample for component $k(s)$ only if $\lVert s-c_{k(s)}\rVert\le r$, that is $\lVert s-c_{k(s)}\rVert^2\le r^2$: a sample exactly on the radius **is** counted. Samples outside every radius are counted nowhere.
3. **Covered modes.** Component $k$ is covered when its count is at least $\lceil n/(8K)\rceil$, one eighth of a perfectly fair share.

This returns per-component counts, shape `(K,)`, and the number of covered modes. With $n=512$ and $K=4$, the threshold is $\lceil512/32\rceil=16$ samples. With the blobs' standard deviation $0.2$, a radius $r=0.6$ is three standard deviations.

**What it measures.** The count vector exposes collapse (one large count, the rest near zero) and also imprecision (most samples outside every radius). It does **not** measure fine detail inside a blob, and it needs known centres, which is exactly why this unit trains on a synthetic mixture. Implementing the rule with exact boundary behaviour is p08.

**Worked example.** Take centres $(1,1)$ and $(-1,1)$, $r=0.5$, and samples $(1.5,1)$, $(0,1)$, $(-1,1.2)$. The first is at distance exactly $0.5$ from $(1,1)$: counted. The second is equidistant ($1$ from each): it is assigned to the lower index, $(1,1)$, but $1>0.5$, so it is not counted. The third is counted for $(-1,1)$. Counts: $(1,1)$; with $n=3$, $K=2$ the threshold is $\lceil3/16\rceil=1$, so both modes are covered.

**Checkpoint 4A.** For $n=200$ samples and $K=4$ components, what count does a component need to be covered?

**Checkpoint 4B.** A sample lies at distance exactly $r$ from its nearest centre. Is it counted? What happens if an implementation uses `<` instead of `<=`?

**Collected answers.** $\lceil200/32\rceil=\lceil6.25\rceil=7$. Yes, it is counted; with `<` it would be dropped, so counts on boundary probes come out one too small (the exact defect p08's probes catch).

## 5. Discriminator overpowering

The second classic failure is the opposite imbalance: the discriminator wins outright.

**Symptoms in the trace.** $\mathcal L_D$ falls toward $0$ (real logits large and positive, fake logits large and negative), and the non-saturating $\mathcal L_G=-\text{mean}\log\sigma(\ell_{\text{fake}})$ grows large, because $\ell_{\text{fake}}\ll0$. With the saturating loss the symptom would instead be a generator gradient near $0$ (Session 1, Section 7).

**Causes.** A discriminator learning rate much larger than the generator's, a much larger discriminator, or several discriminator steps per generator step. Once the discriminator separates the two sets with a wide margin, the generator receives gradients only from the steep side of the sigmoid and its samples can freeze in a poor configuration.

**Remedies.** Rebalance the learning rates (or the number of D-steps per G-step), shrink the discriminator, keep the non-saturating generator loss, and re-check sample coverage after the change. Label smoothing and noise injection exist too, but this unit does not use them.

**Telling the two failures apart.**

| Signal | Mode collapse | Discriminator overpowering |
|---|---|---|
| $\mathcal L_D$ window means | moderate, often near $2\log2$ | near $0$ |
| $\mathcal L_G$ window means | moderate | large and growing |
| coverage counts | one or two large counts | poor (samples stranded, often outside every radius) |
| first remedy | lower the generator's learning rate or restore balance, then re-check coverage | lower the discriminator's learning rate or D-steps |

p22 asks you to make this diagnosis from literal traces and choose a remedy.

**Checkpoint 5A.** A trace shows $\mathcal L_D$ window means of $0.01$ and $\mathcal L_G$ window means near $8$. Which failure is this?

**Checkpoint 5B.** Would switching that run to the saturating generator loss help?

**Collected answers.** Discriminator overpowering. No: with fake logits far below zero the saturating gradient $-\sigma(\ell)$ is nearly $0$, so the generator would learn even less; the remedy is to rebalance the players.

## 6. A fixed-seed tiny 2-D GAN training trace

The cell below loads the unit's data through `data/generative_data.py` (which verifies every row's SHA-256 on import), trains the unit's MLP generator and discriminator for **400** alternating steps on the 256 training rows with equal learning rates `1e-3`, and reports 50-step window means of both losses every 100 steps together with the coverage counts of 512 fixed-noise samples at radius $0.6$.

**What to look for.** In the reference run the discriminator loss stays between about $1.0$ and $1.35$, near $2\log2$, and the generator loss stays moderate: by the trace alone the game looks balanced. The coverage counts tell a different story: nearly all counted samples sit on **one** blob. This is mode collapse with a healthy-looking trace.

The run is lesson-scale and deliberately differs from p17's pinned protocol (step count, learning rates, and noise seeds); p17 fixes its own contract and certificates.

**Checkpoint 6A.** Why does the cell draw the evaluation noise from a different seeded generator than the training noise?

**Checkpoint 6B.** If the coverage counts were $(0,0,8,192)$ for 512 samples, how many modes are covered, and what fraction of samples fall inside any radius?

**Collected answers.** So the evaluation samples are reproducible and independent of how many training steps consumed the training noise stream. One mode ($8<16\le192$), and $200/512\approx0.39$ of the samples fall inside a radius.

In [ ]:
import importlib.util
from pathlib import Path

import torch

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)
from torch import nn
import torch.nn.functional as F


def make_generator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))


def make_discriminator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))


def loss_d(real_logits, fake_logits):
    return (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))


def loss_g(fake_logits):
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


def run_gan(lr_g, lr_d, steps, noise_seed):
    """Lesson-scale alternating training on the 256 training rows; returns (G, D, trace)."""
    real = generative_data.train_tensor("mixture2d")
    torch.manual_seed(SEED)
    G, D = make_generator(), make_discriminator()
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d)
    noise = torch.Generator().manual_seed(noise_seed)
    trace = []
    for _ in range(steps):
        z = torch.randn(real.shape[0], 2, generator=noise)
        opt_d.zero_grad(set_to_none=True)                      # D-step
        d_value = loss_d(D(real), D(G(z).detach()))
        d_value.backward()
        opt_d.step()
        opt_g.zero_grad(set_to_none=True)                      # G-step
        g_value = loss_g(D(G(z)))
        g_value.backward()
        opt_g.step()
        trace.append((d_value.item(), g_value.item()))
    return G, D, trace


def coverage_counts(samples, centers, radius):
    d2 = ((samples[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)   # (n, K)
    nearest = d2.argmin(dim=1)                                           # ties -> lowest index
    inside = d2.min(dim=1).values <= radius ** 2
    return torch.bincount(nearest[inside], minlength=centers.shape[0])


def window_means(trace, width=50):
    return [tuple(round(sum(row[i] for row in trace[a:a + width]) / width, 3) for i in range(2))
            for a in range(0, len(trace), width * 2)]

G, D, trace = run_gan(lr_g=1e-3, lr_d=1e-3, steps=400, noise_seed=SEED + 10)
print("(D-loss, G-loss) 50-step window means every 100 steps:", window_means(trace))
with torch.no_grad():
    samples = G(torch.randn(512, 2, generator=torch.Generator().manual_seed(SEED + 11)))
counts = coverage_counts(samples, generative_data.mixture_centers(), radius=0.6)
need = -(-512 // (8 * 4))
print("coverage counts:", counts.tolist(), " covered modes:", int((counts >= need).sum()), " threshold:", need)

## 7. Worked laboratory: diagnosing an overpowering discriminator

**Worked exam-style example (diagnosis from a trace).** A teammate reports these 50-step window means (D-loss, G-loss): $(0.35,3.6)$, $(0.006,8.3)$, $(0.05,8.1)$, $(0.23,5.7)$, and coverage counts $(0,0,255,34)$ for 512 samples at $r=0.6$.

1. *Balance:* $\mathcal L_D$ is far below $2\log2$ and $\mathcal L_G$ is large: the discriminator dominates.
2. *Coverage:* threshold $16$; two components reach it, two have $0$; $289/512\approx56\%$ of samples are inside a radius.
3. *Diagnosis:* discriminator overpowering with partial coverage. *Remedy:* lower the discriminator's learning rate (or raise the generator's) and re-check both the window means and the coverage counts; do not switch to the saturating loss.

The cell below reproduces this kind of run: the same networks, but the discriminator learns with `lr=1e-2` and the generator with `lr=1e-4`. In the reference run the discriminator loss drops below $0.1$ within the first hundred steps, and the generator loss rises above $5$.

**Worked exam-style example (one alternating step by hand).** Suppose a 1-D discriminator outputs logits $\ell_{\text{real}}=1$ and $\ell_{\text{fake}}=-1$ on one real and one fake row. Then $\mathcal L_D=\log(1+e^{-1})+\log(1+e^{-1})=2(0.313262)=0.626523$ and the non-saturating $\mathcal L_G=\log(1+e^{1})=1.313262$. If a D-step makes the logits $(2,-2)$, $\mathcal L_D$ falls to $2\log(1+e^{-2})=0.253856$ and $\mathcal L_G$ rises to $\log(1+e^{2})=2.126928$: the players' losses move in opposite directions by design.

**Checkpoint 7A.** In the hand example, compute $\mathcal L_D$ when both logits are $0$, and compare it with $2\log2$.

**Checkpoint 7B.** In the overpowered run, which single hyperparameter change would you try first, and what would you check afterwards?

**Collected answers.** $2\log2=1.386294$: identical, because a zero logit is the confused discriminator. Lower the discriminator's learning rate (for example to the generator's), then check that the $\mathcal L_D$ window means rise toward $2\log2$ and that the coverage counts improve; a better-looking trace alone is not enough.

In [ ]:
import importlib.util
from pathlib import Path

import torch

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)
from torch import nn
import torch.nn.functional as F


def make_generator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))


def make_discriminator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))


def loss_d(real_logits, fake_logits):
    return (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))


def loss_g(fake_logits):
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


def run_gan(lr_g, lr_d, steps, noise_seed):
    """Lesson-scale alternating training on the 256 training rows; returns (G, D, trace)."""
    real = generative_data.train_tensor("mixture2d")
    torch.manual_seed(SEED)
    G, D = make_generator(), make_discriminator()
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d)
    noise = torch.Generator().manual_seed(noise_seed)
    trace = []
    for _ in range(steps):
        z = torch.randn(real.shape[0], 2, generator=noise)
        opt_d.zero_grad(set_to_none=True)                      # D-step
        d_value = loss_d(D(real), D(G(z).detach()))
        d_value.backward()
        opt_d.step()
        opt_g.zero_grad(set_to_none=True)                      # G-step
        g_value = loss_g(D(G(z)))
        g_value.backward()
        opt_g.step()
        trace.append((d_value.item(), g_value.item()))
    return G, D, trace


def coverage_counts(samples, centers, radius):
    d2 = ((samples[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)   # (n, K)
    nearest = d2.argmin(dim=1)                                           # ties -> lowest index
    inside = d2.min(dim=1).values <= radius ** 2
    return torch.bincount(nearest[inside], minlength=centers.shape[0])


def window_means(trace, width=50):
    return [tuple(round(sum(row[i] for row in trace[a:a + width]) / width, 3) for i in range(2))
            for a in range(0, len(trace), width * 2)]

G, D, trace = run_gan(lr_g=1e-4, lr_d=1e-2, steps=400, noise_seed=SEED + 10)
print("(D-loss, G-loss) 50-step window means every 100 steps:", window_means(trace))
with torch.no_grad():
    samples = G(torch.randn(512, 2, generator=torch.Generator().manual_seed(SEED + 11)))
    real_prob = torch.sigmoid(D(generative_data.train_tensor("mixture2d"))).mean().item()
    fake_prob = torch.sigmoid(D(samples)).mean().item()
print(f"mean D(real) = {real_prob:.3f}   mean D(fake) = {fake_prob:.3f}")
print("coverage counts:", coverage_counts(samples, generative_data.mixture_centers(), radius=0.6).tolist())

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Missing `detach` in the D-step.* The generator acquires gradients from the discriminator's loss. Fix: `D(G(z).detach())`, and probe that the generator's `.grad` fields are `None` after `d_step`.
2. *One optimizer for both networks.* Then each step moves both players toward the same loss. Fix: two optimizers, each over one network's parameters.
3. *Forgetting `zero_grad`.* Gradients accumulate across steps and across players. Fix: each helper begins with `zero_grad(set_to_none=True)`.
4. *Held-out rows as "real" data.* Held-out rows evaluate; they never enter `D(real)` during training. Fix: train on `train_rows()` only and hash-check what the discriminator's `forward` receives (p17).
5. *Global-RNG noise inside helpers.* Results then depend on every earlier random call. Fix: one dedicated seeded generator in the training loop; pass `z` in.
6. *Reading losses as quality.* Fix: certify with coverage counts and window means together.
7. *Radius off-by-one.* `<` instead of `<=` at the radius, or counting samples outside every radius. Fix: compare squared distances with `<=` and count only inside the radius.

**Exam connections.** p07 implements `gan_step` with the pinned helpers and proves by probes which network each half changes; p08 implements `mode_coverage` with exact boundary probes; p17 trains the unit's GAN and certifies a discriminator-loss band around $2\log2$, a finite generator loss, and a coverage count; p22 diagnoses mode collapse versus overpowering from literal traces; p14 (proof) connects the trace's reference value $2\log2$ to the JS optimum.

**Checkpoint 8A.** Which probe detects a missing `detach`, and why does a weight comparison not detect it?

**Checkpoint 8B.** Which practice certifies a coverage count after training?

**Collected answers.** Checking that every generator parameter's `.grad` is `None` right after `d_step`; the weights are unchanged with or without the bug, because `opt_d` never updates generator weights. p17.

## 9. Forward-only deeper topic: convolutional GANs (DCGAN)

Forward-only deeper topic.
Image GANs replace the MLP generator with a stack of **transposed convolutions**, layers that map a small feature map to a larger one, so that a noise vector grows into an image; the discriminator is a convolutional classifier. This design is usually called DCGAN, and it adds batch normalization and specific activation choices to stabilize training.
Book 1 never taught transposed convolution, so this unit keeps both networks as MLPs on 2-D data.
None of these architectures is a prerequisite, a permitted shortcut, or an assessed API in B2-023.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Does any B2-023 practice require a transposed convolution?

**Checkpoint 9B.** Which network in a DCGAN grows a small input into a large output?

**Collected answers.** No; it is forward-only and not assessed. The generator, through its transposed convolutions.